In [ ]:
# ============================================================
# DESIGN MAPS FOR STORED AND LEAKED CO2 MASS
# ============================================================
# This block generates decision-oriented ANN response maps for:
# - Stored CO2 mass
# - Leaked CO2 mass
#
# Required objects from previous ANN/PDP workflow:
# - df
# - X_COLS
# - predict_stored_from_raw
# - predict_leaked_from_raw
# - FIGURES_DIR
# - TABLES_DIR
#
# Notes:
# - Uses leakage-relevant original dataset only.
# - No secure cases.
# - No oversampling.
# - No df_balanced.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap


# ============================================================
# SETTINGS AND DESIGN-MAP DATASET
# ============================================================

DISTANCE_FACTOR_M = 20.0

df_map = df[df["leak_category"] != "secure"].copy()
df_map.reset_index(drop=True, inplace=True)

print("\n=== DESIGN MAP DATASET ===")
print("Dataset shape:", df_map.shape)
print("\nClass distribution:")
print(df_map["leak_category"].value_counts())

label_map = {
    "PERM_permeability": "Reservoir permeability (mD)",
    "porosity": "Reservoir porosity (-)",
    "PERM_dist_to_well_in_grid": "Injector-to-well distance (m)",
    "PERM_perm_factor_of_failed_well": "Adjacent-well failure factor (-)",
    "PERM_caprock_perm": "Caprock permeability (mD)",
    "aquifer_porosity": "Aquifer porosity (-)",
    "aquifer_permeability": "Aquifer permeability (mD)",
}

cmap_blue_white_red = LinearSegmentedColormap.from_list(
    "blue_white_red",
    [
        (0.0, "#08306B"),
        (0.40, "#4292C6"),
        (0.50, "#FFFFFF"),
        (0.70, "#EE5A49"),
        (1.0, "#7F0000"),
    ],
    N=256
)


# ============================================================
# ANN RESPONSE SURFACE WITH DESIGN ZONES
# ============================================================

def ann_surface_with_zones(
    ax,
    df,
    input_columns,
    predict_function,
    var_x,
    var_y,
    title,
    thresholds=None,
    zone_labels=None,
    label_map=None,
    nx=160,
    ny=160,
    cmap=cmap_blue_white_red,
    convert_distance_to_m=True,
    distance_factor_m=20.0
):
    """
    Generate an ANN-based response surface for two selected variables,
    while keeping all remaining variables fixed at their mean values.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axes object used for plotting.
    df : pandas.DataFrame
        Dataset used to define variable ranges and mean base point.
    input_columns : list
        Input feature names used by the ANN model.
    predict_function : callable
        Function that receives raw X and returns predictions in physical units.
    var_x : str
        Variable shown on the x-axis.
    var_y : str
        Variable shown on the y-axis.
    title : str
        Plot title.
    thresholds : list or None
        Optional contour threshold values.
    zone_labels : list or None
        Optional list of tuples: [(label, (x_position, y_position)), ...].
    label_map : dict or None
        Dictionary for pretty axis labels.
    nx, ny : int
        Grid resolution.
    cmap : matplotlib colormap
        Colormap used for filled contours.
    convert_distance_to_m : bool
        If True, converts grid-cell distance to meters for plotting.
    distance_factor_m : float
        Conversion factor from grid cells to meters.

    Returns
    -------
    cf : contourf object
    surface_long : pandas.DataFrame
        Long-format surface table with x, y, and predicted values.
    Z : numpy.ndarray
        Prediction surface matrix.
    xs_plot : numpy.ndarray
        x-axis values used in the plot.
    ys_plot : numpy.ndarray
        y-axis values used in the plot.
    """

    if var_x not in input_columns:
        raise ValueError(f"{var_x} not found in input_columns.")

    if var_y not in input_columns:
        raise ValueError(f"{var_y} not found in input_columns.")

    idx_x = input_columns.index(var_x)
    idx_y = input_columns.index(var_y)

    pretty_x = label_map.get(var_x, var_x) if label_map else var_x
    pretty_y = label_map.get(var_y, var_y) if label_map else var_y

    base_point = df[input_columns].mean().values

    xs_model = np.linspace(df[var_x].min(), df[var_x].max(), nx)
    ys_model = np.linspace(df[var_y].min(), df[var_y].max(), ny)

    Z = np.zeros((ny, nx))

    for i, xv in enumerate(xs_model):
        for j, yv in enumerate(ys_model):
            x_input = base_point.copy()
            x_input[idx_x] = xv
            x_input[idx_y] = yv
            Z[j, i] = predict_function(x_input.reshape(1, -1))[0]

    xs_plot = xs_model.copy()
    ys_plot = ys_model.copy()

    if convert_distance_to_m and var_x == "PERM_dist_to_well_in_grid":
        xs_plot = xs_model * distance_factor_m
        pretty_x = "Injector-to-well distance (m)"

    if convert_distance_to_m and var_y == "PERM_dist_to_well_in_grid":
        ys_plot = ys_model * distance_factor_m
        pretty_y = "Injector-to-well distance (m)"

    X_grid_plot, Y_grid_plot = np.meshgrid(xs_plot, ys_plot)

    surface_long = pd.DataFrame({
        pretty_x: X_grid_plot.ravel(),
        pretty_y: Y_grid_plot.ravel(),
        "Predicted value (t)": Z.ravel(),
    })

    cf = ax.contourf(
        xs_plot,
        ys_plot,
        Z,
        levels=60,
        cmap=cmap
    )

    ax.contour(
        xs_plot,
        ys_plot,
        Z,
        levels=12,
        colors="black",
        linewidths=0.35,
        alpha=0.35
    )

    if thresholds is not None:
        for threshold in thresholds:
            cs = ax.contour(
                xs_plot,
                ys_plot,
                Z,
                levels=[threshold],
                colors="yellow",
                linewidths=2.0
            )

            ax.clabel(
                cs,
                fmt={threshold: f"{threshold:,.0f}"},
                fontsize=9,
                colors="yellow"
            )

    if zone_labels is not None:
        for text, pos in zone_labels:
            ax.text(
                pos[0],
                pos[1],
                text,
                fontsize=11,
                color="white",
                weight="bold",
                ha="center",
                va="center",
                bbox=dict(
                    boxstyle="round,pad=0.30",
                    fc="black",
                    ec="none",
                    alpha=0.30
                )
            )

    ax.set_title(title, fontsize=14)
    ax.set_xlabel(pretty_x, fontsize=12)
    ax.set_ylabel(pretty_y, fontsize=12)
    ax.tick_params(axis="both", labelsize=11)

    return cf, surface_long, Z, xs_plot, ys_plot


# ============================================================
# STORED CO2 MASS DESIGN MAP
# ============================================================

fig, ax = plt.subplots(figsize=(7.0, 5.8), dpi=600)

cf_stored, stored_surface_long, Z_stored, xs_stored, ys_stored = ann_surface_with_zones(
    ax=ax,
    df=df_map,
    input_columns=X_COLS,
    predict_function=predict_stored_from_raw,
    var_x="PERM_permeability",
    var_y="porosity",
    title="Stored CO₂ Mass Map",
    thresholds=[1.1e6, 1.4e6],
    zone_labels=[
        (
            "LOW STORAGE",
            (
                df_map["PERM_permeability"].min() + 35,
                df_map["porosity"].min() + 0.015
            )
        ),
        (
            "MEDIUM STORAGE",
            (
                df_map["PERM_permeability"].mean(),
                df_map["porosity"].mean()
            )
        ),
        (
            "HIGH STORAGE",
            (
                df_map["PERM_permeability"].min() + 45,
                df_map["porosity"].max() - 0.018
            )
        )
    ],
    label_map=label_map,
    convert_distance_to_m=True,
    distance_factor_m=DISTANCE_FACTOR_M
)

cbar_stored = fig.colorbar(cf_stored, ax=ax, fraction=0.046, pad=0.04)
cbar_stored.set_label("Stored CO₂ mass (t)", fontsize=11)
cbar_stored.ax.tick_params(labelsize=10)

plt.tight_layout()

stored_fig_path = FIGURES_DIR / "Figure_13_StoredCO2Mass_Map.png"
fig.savefig(stored_fig_path, dpi=600, bbox_inches="tight")

plt.show()
plt.close(fig)

stored_surface_long.to_csv(
    TABLES_DIR / "Figure_13_StoredCO2Mass_Map_surface_values.csv",
    index=False
)

print(f"Saved stored CO2 mass map: {stored_fig_path}")
print(f"Saved stored surface values: {TABLES_DIR / 'Figure_13_StoredCO2Mass_Map_surface_values.csv'}")


# ============================================================
# LEAKED CO2 MASS DESIGN MAP
# ============================================================

fig, ax = plt.subplots(figsize=(7.0, 5.8), dpi=600)

distance_min_m = df_map["PERM_dist_to_well_in_grid"].min() * DISTANCE_FACTOR_M
distance_max_m = df_map["PERM_dist_to_well_in_grid"].max() * DISTANCE_FACTOR_M
distance_mean_m = df_map["PERM_dist_to_well_in_grid"].mean() * DISTANCE_FACTOR_M

cf_leaked, leaked_surface_long, Z_leaked, xs_leaked, ys_leaked = ann_surface_with_zones(
    ax=ax,
    df=df_map,
    input_columns=X_COLS,
    predict_function=predict_leaked_from_raw,
    var_x="PERM_permeability",
    var_y="PERM_dist_to_well_in_grid",
    title="Leaked CO₂ Mass Map",
    thresholds=[1e4, 3e4],
    zone_labels=[
        (
            "LOW LEAKAGE",
            (
                df_map["PERM_permeability"].min() + 45,
                distance_max_m - 15
            )
        ),
        (
            "MEDIUM LEAKAGE",
            (
                df_map["PERM_permeability"].mean(),
                distance_mean_m
            )
        ),
        (
            "HIGH LEAKAGE",
            (
                df_map["PERM_permeability"].min() + 45,
                distance_min_m + 15
            )
        )
    ],
    label_map=label_map,
    convert_distance_to_m=True,
    distance_factor_m=DISTANCE_FACTOR_M
)

cbar_leaked = fig.colorbar(cf_leaked, ax=ax, fraction=0.046, pad=0.04)
cbar_leaked.set_label("Leaked CO₂ mass (t)", fontsize=11)
cbar_leaked.ax.tick_params(labelsize=10)

plt.tight_layout()

leaked_fig_path = FIGURES_DIR / "Figure_14_LeakedCO2Mass_Map.png"
fig.savefig(leaked_fig_path, dpi=600, bbox_inches="tight")

plt.show()
plt.close(fig)

leaked_surface_long.to_csv(
    TABLES_DIR / "Figure_14_LeakedCO2Mass_Map_surface_values.csv",
    index=False
)

print(f"Saved leaked CO2 mass map: {leaked_fig_path}")
print(f"Saved leaked surface values: {TABLES_DIR / 'Figure_14_LeakedCO2Mass_Map_surface_values.csv'}")


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\nDesign-map generation completed successfully.")
print("Generated:")
print(f"- {stored_fig_path}")
print(f"- {leaked_fig_path}")